# 03 - Evaluacion

Mide la exactitud por ejecucion (EX ponderada) sobre la validacion propia:
40 preguntas escritas a mano sobre 2 esquemas que no aparecen en el entrenamiento.

Se corren dos versiones con el mismo prompt y el mismo criterio:

- **baseline**: el modelo base, `usar_adapter: false`
- **adapter**: el modelo con el adapter LoRA entrenado

La diferencia entre ambas es lo que mide el aporte del fine-tuning, y vale 10 de
los 40 puntos de la evaluacion oculta.

In [ ]:
!pip install -q -r requirements.txt
!bash scripts/mysql_setup.sh colab
!python scripts/cargar_esquemas.py --dir datos/validacion/esquemas --poblar

In [ ]:
import os

os.environ['MYSQL_HOST'] = '127.0.0.1'
os.environ['MYSQL_PORT'] = '3306'
os.environ['MYSQL_USER'] = 'lector'
os.environ['MYSQL_PASSWORD'] = 'lector'

## 1. El criterio de comparacion

`coincide()` replica las reglas del enunciado: se comparan resultados y no texto,
el orden de las filas solo cuenta si la pregunta lo pide, los numeros se redondean
a 2 decimales y una columna adicional no penaliza.

Si esa funcion se equivoca, todos los numeros de este notebook se equivocan con
ella, asi que se prueba primero.

In [ ]:
!python scripts/test_comparacion.py
!python scripts/test_extraccion.py

## 2. Baseline: el modelo sin adapter

In [ ]:
!python scripts/evaluar.py --etiqueta baseline --sin-adapter

## 3. El modelo con el adapter LoRA

Requiere haber corrido `02_entrenamiento.ipynb`, que deja los pesos en `adapter/`.

In [ ]:
!python scripts/evaluar.py --etiqueta adapter

## 4. Comparacion

EX ponderada = suma de los pesos acertados entre la suma de todos los pesos
(facil 1, media 2, dificil 3).

In [ ]:
import pandas as pd

df = pd.read_csv('resultados/validacion_propia.csv')

def ex(grupo):
    return (grupo['peso'] * grupo['ok']).sum() / grupo['peso'].sum()

global_ex = df.groupby('etiqueta').apply(ex).rename('EX')
por_nivel = df.groupby(['etiqueta', 'nivel']).apply(ex).unstack().round(3)
display(global_ex.round(3))
display(por_nivel)

if {'baseline', 'adapter'} <= set(global_ex.index):
    mejora = global_ex['adapter'] - global_ex['baseline']
    print('mejora del adapter sobre el baseline: %+.3f' % mejora)
    print('umbral del enunciado para los 10 puntos: +0.150')

In [ ]:
# Preguntas que el adapter arregla y preguntas que rompe.
pivote = df.pivot_table(index='id', columns='etiqueta', values='ok')
if {'baseline', 'adapter'} <= set(pivote.columns):
    arregladas = pivote[(pivote['baseline'] == 0) & (pivote['adapter'] == 1)].index.tolist()
    rotas = pivote[(pivote['baseline'] == 1) & (pivote['adapter'] == 0)].index.tolist()
    print('arregla:', arregladas)
    print('rompe  :', rotas)

In [ ]:
# Tiempos: el contrato permite 30 s por pregunta y 5 min de carga.
display(df.groupby('etiqueta')['segundos'].agg(['mean', 'max']).round(2))
display(df[df['error'].notna()].groupby('etiqueta').size().rename('consultas que no ejecutan'))

## 5. Siguiente paso

El analisis de errores por categoria y el experimento de inferencia
(autocorreccion con `ejecutar`) estan en `scripts/analizar_errores.py`, que lee
este mismo `resultados/validacion_propia.csv`.